#### 投研环境中， 自行合成数据和文华财经对比
- 使用不复权数据

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
import pandas as pd

In [2]:
from config.contract import TRADING_TIME_MAPPING
from kdutils.data import fetch_local_market1
from lib.attr001.ftd001 import filter_trading_time

/workspace/worker/env/lingua/lib/python3.8/site-packages/Finance_Jindowin-1.5.4-py3.8.egg/jdw/__init__.py:11: UserWarning: if use distributed calculating, please configure MQ_URL
  warnings.warn('if use distributed calculating, please configure MQ_URL')
/workspace/worker/env/lingua/lib/python3.8/site-packages/Finance_Jindowin-1.5.4-py3.8.egg/jdw/__init__.py:15: UserWarning: if use distributed calculating, please configure NTN_URL
  warnings.warn('if use distributed calculating, please configure NTN_URL')
/workspace/worker/env/lingua/lib/python3.8/site-packages/Finance_Jindowin-1.5.4-py3.8.egg/jdw/__init__.py:19: UserWarning: if use memory database, please configure KN_MG
  warnings.warn('if use memory database, please configure KN_MG')
/workspace/worker/env/lingua/lib/python3.8/site-packages/Finance_Jindowin-1.5.4-py3.8.egg/jdw/__init__.py:27: UserWarning: if use trader, please configure ATL_URL
  warnings.warn('if use trader, please configure ATL_URL')
/workspace/worker/env/lingua/lib

In [3]:
code = 'AP'
trading_sessions = TRADING_TIME_MAPPING[code]
begin_date = '2026-06-01' #'2025-01-01'
end_data = '2026-06-30'#'2026-01-01'

In [4]:
keep_columns = ['trade_time','code','symbol', 'open','high','low','close','vwap','openint','volume','value']

In [7]:
research_data = fetch_local_market1(base_path=os.environ['BAR_FUT_DIRS'],
                                       begin_date=begin_date,
                                       end_date=end_data,
                                       keep_symbol=True,
                                       codes=[code],
                                       method=None) ## 严禁使用复权价格
research_data = research_data[keep_columns]
research_data['trade_time'] = pd.to_datetime(research_data['trade_time'])
research_data = research_data.set_index(['trade_time','code'])
research_data = filter_trading_time(data=research_data, trading_sessions=trading_sessions).sort_values(by=['trade_time','code'])

/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260601/AP610_20260601.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260602/AP610_20260602.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260603/AP610_20260603.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260604/AP610_20260604.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260605/AP610_20260605.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260608/AP610_20260608.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260609/AP610_20260609.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260610/AP610_20260610.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260611/AP610_20260611.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260612/AP610_20260612.feather
/workspace/data/dev/kd/intelkit/records/raw_data/cn_futures/20260615/AP610_20260

In [8]:
research_data[research_data['trade_time'] > '2026-06-26 22:55:00'].head(40)

,trade_time,code,symbol,open,high,low,close,vwap,openint,volume,value
4301,2026-06-29 09:00:00,AP,AP610,7665.0,7666.0,7618.0,7640.0,7644.848473,114040.0,5108.0,39049886.0
4302,2026-06-29 09:01:00,AP,AP610,7640.0,7647.0,7625.0,7640.0,7637.254916,113086.0,2848.0,21750902.0
4303,2026-06-29 09:02:00,AP,AP610,7639.0,7676.0,7637.0,7676.0,7657.049689,112380.0,2254.0,17258990.0
4304,2026-06-29 09:03:00,AP,AP610,7678.0,7680.0,7664.0,7674.0,7667.637427,112035.0,2394.0,18356324.0
4305,2026-06-29 09:04:00,AP,AP610,7671.0,7672.0,7647.0,7647.0,7660.832954,111764.0,1317.0,10089317.0
4306,2026-06-29 09:05:00,AP,AP610,7649.0,7664.0,7647.0,7663.0,7662.654966,111543.0,1339.0,10260295.0
4307,2026-06-29 09:06:00,AP,AP610,7661.0,7670.0,7650.0,7670.0,7652.000000,111522.0,1046.0,8003992.0
4308,2026-06-29 09:07:00,AP,AP610,7669.0,7673.0,7668.0,7668.0,7675.411844,111492.0,743.0,5702831.0
4309,2026-06-29 09:08:00,AP,AP610,7668.0,7675.0,7658.0,7674.0,7653.000000,111359.0,711.0,5441283.0
4310,2026-06-29 09:09:00,AP,AP610,7675.0,7681.0,7672.0,7675.0,7685.636210,111236.0,1182.0,9084422.0


##### 校验数据的周期

- 1. 夜盘开盘时间
- 2. 日盘开盘时间
- 3. 跨日常休息日
- 4. 跨节假休息日
